In [1]:
import pandas as pd
import numpy as np
import math

#  Charger le dataset
df = pd.read_csv("data.csv")

# Supprimer les colonnes inutiles
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Transformer la cible :
# M = Malignant -> 1
# B = Benign -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Séparer X et y
X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(int)


#  Train / Test split
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]


#  Naive Bayes Gaussien from scratch
class GaussianNaiveBayes:
    def fit(self, X, y):
        # récupérer les classes existantes : 0 et 1
        self.classes = np.unique(y)

        # dictionnaires pour stocker moyenne, variance et probabilité
        self.mean = {}
        self.var = {}
        self.prior = {}

        for c in self.classes:
            # sélectionner les lignes appartenant à la classe c
            X_c = X[y == c]

            # moyenne de chaque variable pour cette classe
            self.mean[c] = X_c.mean(axis=0)

            # variance de chaque variable pour cette classe
            self.var[c] = X_c.var(axis=0) + 1e-8

            # probabilité de la classe
            self.prior[c] = len(X_c) / len(X)

    def gaussian_probability(self, x, mean, var):
        # formule de la densité gaussienne
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * math.pi * var)

        return numerator / denominator

    def predict_one(self, x):
        probabilities = {}

        for c in self.classes:
            # commencer par la probabilité de la classe
            probabilities[c] = math.log(self.prior[c])

            # calculer les probabilités gaussiennes pour chaque variable
            probs = self.gaussian_probability(x, self.mean[c], self.var[c])

            # utiliser log pour éviter les très petits nombres
            probabilities[c] += np.sum(np.log(probs + 1e-8))

        # choisir la classe avec la plus grande probabilité
        return max(probabilities, key=probabilities.get)

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])


#  Entraîner le modèle
model = GaussianNaiveBayes()
model.fit(X_train, y_train)


#  Faire les prédictions
y_pred = model.predict(X_test)


#  Évaluation simple 
correct = 0
incorrect = 0

for i in range(len(y_test)):
    if y_test[i] == y_pred[i]:
        correct += 1
    else:
        incorrect += 1

print("Évaluation Naive Bayes :")
print("Bonnes prédictions :", correct)
print("Mauvaises prédictions :", incorrect)
print("Total :", len(y_test))


Évaluation Naive Bayes :
Bonnes prédictions : 106
Mauvaises prédictions : 8
Total : 114
